# SOP Debugging Lab: From Product Terms to Tree Environments

This executable lab uses one deterministic Li.W.2024 spin--boson case. Run it top to bottom with `RUN_BREAKPOINTS = False`; set the flag to `True` only when you are ready to stop at the production breakpoint named in an investigation.

**Scientific boundary.** The formal benchmark quantity is `local_effective_1site_apply_all_nodes`: one-site local effective-Hamiltonian actions collected across all active tree nodes. It is neither a complete TDVP propagation step nor a full-state `H|psi>` application.


In [ ]:
from pathlib import Path
import inspect
import numpy as np

from benchmarks.sop_debugging_lab import (
    build_lab_case,
    compare_cache_modes,
    summarize_strict_cache,
    summarize_symbolic_terms,
    summarize_tree_local_ops,
    summarize_ttno,
)

RUN_BREAKPOINTS = False
ctx = build_lab_case()
tree, terms, psi = ctx["tree"], ctx["terms"], ctx["psi"]
sop, ttno = ctx["sop"], ctx["ttno"]
print(ctx["metadata"])


# Investigation 1 — End-to-End Experiment Map

## 1. Supervisor question
Where does one symbolic Hamiltonian term travel before it becomes a plotted scaling point?

## 2. Formula
$\text{Op list} \rightarrow \{\text{SOP},\text{TTNO}\} \rightarrow y_m \rightarrow \text{snapshot} \rightarrow \bar y \rightarrow t(x)=Cx^\alpha \rightarrow \text{figure}$.

## 3. Runnable observation
The next cell prints the actual case metadata and every named stage in order.

## 4. Exact breakpoint
In `benchmarks/benchmark_li2024_operator_env.py::run_li2024_point`, stop immediately after `_metadata(...)` returns and before `_ttno_active_scope_action(...)`.

## 5. Inspect
Inspect `terms`, `sop.n_terms`, `ttno.bond_dims`, `active_nodes`, `metadata["quantity"]`, and later each row's `method`.

## 6. Answer prompt
**Your answer:** Which stages change the physics object, and which stages only serialize, aggregate, fit, or plot it? ______

## 7. Expected observation
The fixed case has 13 symbolic/SOP terms and quantity `local_effective_1site_apply_all_nodes`; representation changes occur before timing, while snapshot/finalizer/plotter stages preserve the recorded measurement.

## 8. Three-sentence oral explanation
We build one Hamiltonian and one TTNS, then represent the Hamiltonian as flat SOP terms or a compressed TTNO. Each method times the same all-node local-action quantity and writes one recoverable task result. The finalizer aggregates completed rows, fits power laws, and plots them without redefining the measured object.


In [ ]:
pipeline = [
    "symbolic Op terms",
    "Li2024 BasisTree + TTNS",
    "SOPBaselineOperator or TTNO",
    "all-node local effective actions",
    "per-task atomic NPZ snapshot",
    "raw/summary/fits CSV",
    "three-panel figure",
]
if RUN_BREAKPOINTS:
    breakpoint()
print(" -> ".join(pipeline))
print("case metadata:", ctx["metadata"])


# Investigation 2 — `Op` and `SOPTerm`

## 1. Supervisor question
How does the code represent one coefficient times a product of local operators inside a Hamiltonian sum?

## 2. Formula
$$H = \\sum_l c_l \prod_\kappa h_l^{(\kappa)}.$$ 

## 3. Runnable observation
The next cell prints a real symbolic `Op` summary beside its flattened `SOPTerm` coefficient and nontrivial-node count.

## 4. Exact breakpoint
In `renormalizer/tn/sop_baseline.py::SOPTerm.from_op`, stop on `local_ops, coeff = _split_op_by_tree_node(op, basis)`.

## 5. Inspect
Inspect `op.symbol`, `op.dofs`, `op.factor`, `coeff`, `local_ops`, and `len(local_ops)`.

## 6. Answer prompt
**Your answer:** What is stored once as `c_l`, and what is stored by tree-node index as the factors $h_l^{(\kappa)}$? ______

## 7. Expected observation
There are 13 product terms; a coupling term spans spin and boson degrees of freedom but becomes one scalar coefficient plus only its non-identity node-local factors.

## 8. Three-sentence oral explanation
An `Op` is one symbolic product term with a scalar factor and labeled degrees of freedom. `SOPTerm.from_op` removes that scalar into `coeff` and groups the remaining non-identity factors by tree node. The Hamiltonian remains a flat sum of 13 independent product terms rather than an operator-bond network.


In [ ]:
symbolic_rows = summarize_symbolic_terms(terms, sop)
if RUN_BREAKPOINTS:
    breakpoint()
print("n symbolic terms:", len(symbolic_rows), "n SOP terms:", sop.n_terms)
print("first term:", symbolic_rows[0])
print("spin-boson coupling term:", symbolic_rows[-1])


# Investigation 3 — Splitting a Product Term Across Tree Nodes

## 1. Supervisor question
How does one product term learn which factors act on which nodes of a non-linear BasisTree?

## 2. Formula
$\operatorname{node}(d)=\texttt{BasisTree.dof2idx}[d]$, and missing entries in `local_ops` mean identity.

## 3. Runnable observation
The next cell prints every tree node for one spin--boson coupling term, including its dofs, physical bond dimensions, and identity/nontrivial status.

## 4. Exact breakpoint
In `renormalizer/tn/sop_baseline.py::_split_op_by_tree_node`, stop immediately after `node_indices = {basis.dof2idx[dof] for dof in elementary_op.dofs}`.

## 5. Inspect
Inspect `elementary_ops`, `elementary_op.dofs`, `node_indices`, `node_idx`, and the growing `local_ops` mapping.

## 6. Answer prompt
**Your answer:** Which two node indices are nontrivial for the selected coupling term, and why are all other nodes omitted? ______

## 7. Expected observation
The selected `sigma_z x` term has nontrivial factors only on the spin node and one boson-containing node; every omitted node is an implicit identity.

## 8. Three-sentence oral explanation
`Op.split_elementary` first separates elementary operators by degree of freedom. `_split_op_by_tree_node` then maps those dofs through `BasisTree.dof2idx` and multiplies factors that share a node. Sparse `local_ops` storage makes identity action explicit by omission.


In [ ]:
selected_term_index = len(sop.terms) - 1
selected_sop_term = sop.terms[selected_term_index]
tree_local_rows = summarize_tree_local_ops(tree, selected_sop_term)
if RUN_BREAKPOINTS:
    breakpoint()
print("dof -> tree node:", {str(dof): int(idx) for dof, idx in tree.dof2idx.items()})
for row in tree_local_rows:
    print(row)


# Investigation 4 — Symbolic Local Operators to Matrices

## 1. Supervisor question
Where does a symbolic local operator become the numeric matrix that contracts with a physical tensor axis?

## 2. Formula
$[h_l^{(\kappa)}]_{ij}=\langle i|h_l^{(\kappa)}|j\rangle$, with identity $I_d\in\mathbb{R}^{d\times d}$.

## 3. Runnable observation
The next cell selects one real nontrivial local factor, calls its concrete basis `op_mat`, and prints basis dimension, matrix shape, dtype, identity shape, and matrix values.

## 4. Exact breakpoint
In `renormalizer/model/basis.py`, stop inside the selected concrete `BasisHalfSpin.op_mat` or `BasisSHO.op_mat` immediately before it returns the numeric matrix.

## 5. Inspect
Inspect `basis_set.nbas`, `elementary_op.symbol`, `matrix.shape`, `matrix.dtype`, and `np.eye(basis_set.nbas).shape`.

## 6. Answer prompt
**Your answer:** Which physical axis dimension must equal the matrix input dimension, and where is the local scalar factor applied? ______

## 7. Expected observation
The nontrivial matrix is square with size equal to the selected local basis dimension; identity has the same square shape, while `_local_matrix_factors` applies the split local factor exactly once.

## 8. Three-sentence oral explanation
Each concrete `BasisSet` owns the conversion from a symbolic elementary operator to a numeric local matrix. The matrix dimension is fixed by that local basis and must match its TTNS physical axis. The SOP helper caches these matrices so repeated terms avoid repeating symbolic-to-numeric conversion, even though it does not cache contraction environments.


In [ ]:
from renormalizer import Model

matrix_node_idx = next(iter(sorted(selected_sop_term.local_ops)))
basis_node = tree.node_list[matrix_node_idx]
local_op = selected_sop_term.local_ops[matrix_node_idx]
local_model = Model(basis_node.basis_sets, [])
elementary_ops, split_factor = local_op.split_elementary(local_model.dof_to_siteidx)
elementary_op = next(op for op in elementary_ops if not op.is_identity)
site_index = local_model.dof_to_siteidx[elementary_op.dofs[0]]
basis_set = basis_node.basis_sets[site_index]
if RUN_BREAKPOINTS:
    breakpoint()
operator_matrix = basis_set.op_mat(elementary_op)
identity_matrix = np.eye(basis_set.nbas)
print("basis:", type(basis_set).__name__, "dimension:", basis_set.nbas)
print("operator:", elementary_op.to_tuple(), "split factor:", split_factor)
print("operator shape/dtype:", operator_matrix.shape, operator_matrix.dtype)
print("identity shape/dtype:", identity_matrix.shape, identity_matrix.dtype)
print(operator_matrix)


# Investigation 5 — No-Environment SOP Application

## 1. Supervisor question
What work is repeated when flat SOP terms are applied without a contraction environment?

## 2. Formula
$H|\Psi\rangle=\sum_l c_l\left(\prod_\kappa h_l^{(\kappa)}\right)|\Psi\rangle$.

## 3. Runnable observation
The next cell applies one real term with `_apply_term_to_ttns` and prints every source/output node-tensor shape plus the coefficient.

## 4. Exact breakpoint
In `renormalizer/tn/sop_baseline.py::SOPBaselineOperator._apply_term_to_ttns`, stop at the start of `for node_idx, (source_node, target_node) in enumerate(zip(psi, new))`.

## 5. Inspect
Inspect `new`, `node_idx`, `tensor.shape`, `factors`, `physical_axis0`, `term.coeff`, and whether `source_node is psi.root`.

## 6. Answer prompt
**Your answer:** Why does looping over every active node, every SOP term, and a tree traversal lead to cubic mode-count work in the all-node benchmark? ______

## 7. Expected observation
One term creates a state metacopy and visits every tree node, acting only on nontrivial physical axes and multiplying its coefficient at the root; the outer no-env method repeats this full-tree operation independently for all terms.

## 8. Three-sentence oral explanation
The no-environment path applies each flat product term independently and then adds the resulting TTNSs. Local matrices are cached, but branch contractions are not reused between active nodes or terms. For the all-node quantity, active nodes times SOP terms times tree work produces the observed near-cubic mode scaling.


In [ ]:
if RUN_BREAKPOINTS:
    breakpoint()
single_term_state = sop._apply_term_to_ttns(selected_sop_term, psi)
print("term coefficient:", selected_sop_term.coeff)
for node_idx, (source_node, output_node) in enumerate(zip(psi, single_term_state)):
    print(node_idx, source_node.tensor.shape, "->", output_node.tensor.shape,
          "nontrivial:", node_idx in selected_sop_term.local_ops)


# Investigation 6 — Strict MCTDH-Like Directed-Edge Messages

## 1. Supervisor question
What exactly is reused by the strict state-environment SOP baseline, and what is deliberately not shared?

## 2. Formula
$$E_{l,u\\rightarrow v}=\operatorname{contract}\left(\text{bra subtree},\{h_l^{(x)}\},\text{ket subtree}\right).$$

## 3. Runnable observation
The next cell builds the real sweep cache and prints its key schema, cardinality, representative keys, a reverse edge, and all-node output shapes.

## 4. Exact breakpoint
In `benchmarks/benchmark_adaptive_operator_env.py::SOPMCTDHSweepEnvironment.build_env_cache`, stop on the first assignment to `self._env_cache[self._edge_key(...)]`.

## 5. Inspect
Inspect `term_index`, `node`, `node.parent`, `_edge_key(...)`, the cached matrix shape, and the same edge with source/target reversed.

## 6. Answer prompt
**Your answer:** Why does keeping `term_index` in every key provide state-message reuse across active nodes without operator sharing across terms? ______

## 7. Expected observation
The cache contains `2 * (n_nodes - 1) * n_terms` directed messages. Both directions exist for every tree edge and term, and the same built cache serves every active node.

## 8. Three-sentence oral explanation
The strict baseline precontracts bra, one product term, and ket on each directed side of every tree edge. Its key includes the term index, so changing the active node reuses state messages but identical operators from different terms remain separate. This removes one repeated tree factor and yields near-quadratic rather than near-cubic mode scaling.


In [ ]:
strict_env = ctx["strict_env"]
if RUN_BREAKPOINTS:
    breakpoint()
strict_env.build_env_cache()
strict_summary = summarize_strict_cache(strict_env)
strict_keys = sorted(strict_env._env_cache)
first_key = strict_keys[0]
reverse_key = (first_key[1], first_key[0], first_key[2])
strict_actions = strict_env.apply(ctx["active_nodes"])
print(strict_summary)
print("first key / shape:", first_key, strict_env._env_cache[first_key].shape)
print("reverse key exists:", reverse_key in strict_env._env_cache)
print("all-node action shapes:", [action.shape for action in strict_actions])


# Investigation 7 — Strict Term Cache Versus Operator-Signature Cache

## 1. Supervisor question
How can an optimized flat-SOP path share repeated branch operators, and why is it excluded from the formal strict baseline?

## 2. Formula
$K_{\mathrm{term}}=(b,l)$ versus $K_{\mathrm{signature}}=(b,\operatorname{sig}\{h_l^{(x)}\}_{x\in b})$.

## 3. Runnable observation
The next cell builds both cache modes on the same active node and prints counts, keys, and the helper's explicit `signature_metadata` classification.

## 4. Exact breakpoint
In `benchmarks/benchmark_adaptive_operator_env.py::SOPOneSiteEffective._cache_key`, stop before returning the key for `SOP_CACHE_TERM`, then repeat for `SOP_CACHE_SIGNATURE`.

## 5. Inspect
Inspect `child_idx`, `term_index`, `branch_signature(child, term)`, `_env_cache`, and `signature_metadata`.

## 6. Answer prompt
**Your answer:** Which cache may merge two different term indices, and what algorithmic capability does that add? ______

## 7. Expected observation
The signature cache has no more entries than the term cache because repeated branch structures can share one environment. Its metadata says method `sop_env_plus_operator_cache`, purpose `explanatory`, optimized `True`, and strict `False`.

## 8. Three-sentence oral explanation
A term cache identifies environments by branch and term index, so it preserves flat-term separation. A signature cache replaces the term index with the actual branch-operator pattern and can merge repeated structure across terms. That operator reuse is useful but changes the algorithmic identity, so `signature_metadata` explicitly excludes it from the strict three-method comparison.


In [ ]:
if RUN_BREAKPOINTS:
    breakpoint()
cache_comparison = compare_cache_modes(sop, psi)
print("term entries:", cache_comparison["term_entries"])
print("signature entries:", cache_comparison["signature_entries"])
print("term keys:", cache_comparison["term_representative_keys"])
print("signature keys:", cache_comparison["signature_representative_keys"])
print("signature_metadata:", cache_comparison["signature_metadata"])


# Investigation 8 — TTNO Compression and Environment

## 1. Supervisor question
Where is cross-term operator sharing stored in a TTNO, and what does a TTN contraction environment carry?

## 2. Formula
$W^{[v]}_{a_1\ldots a_k,a_p;i_1j_1\ldots i_qj_q}$ stores operator-bond indices $a$ and physical input/output pairs $(i,j)$; an environment edge carries bra, operator, and ket bond axes.

## 3. Runnable observation
The next cell prints the TTNO bond summary, every numeric TTNO tensor shape, and one real `TTNEnviron.environ_parent` shape.

## 4. Exact breakpoint
In `renormalizer/tn/tree.py::TTNEnviron.build_children_environ_node`, stop immediately after `res = oe_contract(...)` and before it is stored on the parent environment node.

## 5. Inspect
Inspect `onode.tensor.shape`, `ttno.bond_dims`, contraction `args`, result `res.shape`, and the three axes of `environ_parent`.

## 6. Answer prompt
**Your answer:** Which TTNO axes encode shared operator structure, and why can their bounded dimensions avoid a separate contraction for every SOP term? ______

## 7. Expected observation
TTNO tensors have physical input/output axes plus virtual operator bonds; a TTN environment message has three bond axes corresponding to bra, operator, and ket. The maximum operator bond is much smaller than the 13-term flat list.

## 8. Three-sentence oral explanation
TTNO construction compresses recurring symbolic subtrees into virtual operator-bond states. `TTNEnviron` contracts bra, TTNO, and ket branches into three-axis boundary tensors for local actions. Bounded operator bonds let one environment network carry shared contributions from many symbolic terms, producing near-linear mode scaling in this case.


In [ ]:
from renormalizer.tn.tree import TTNEnviron

if RUN_BREAKPOINTS:
    breakpoint()
ttno_summary = summarize_ttno(ttno)
ttno_environment = TTNEnviron(psi, ttno)
nonroot_env_node = next(node for node in ttno_environment.node_list if node.parent is not None)
print("TTNO summary:", ttno_summary)
print("TTNO tensor shapes:", [node.tensor.shape for node in ttno.node_list])
print("representative environment-parent shape:", nonroot_env_node.environ_parent.shape)


# Investigation 9 — Numerical Identity and Timing Identity

## 1. Supervisor question
Are all three formal methods numerically equivalent, and exactly what work do their timing fields measure?

## 2. Formula
$\epsilon_m=\|y_m-y_{\mathrm{TTNO}}\|_2/\|y_{\mathrm{TTNO}}\|_2$, where each $y_m$ is the collection of all-node one-site local actions.

## 3. Runnable observation
The next cell runs one small point with exactly the three formal methods and prints method, status, error, environment-build, expression-build, term-loop, apply, and total times.

## 4. Exact breakpoint
In `benchmarks/benchmark_li2024_operator_env.py::run_li2024_point`, stop after `_ttno_active_scope_action(...)` and again after each `_sop_active_scope_action(...)`.

## 5. Inspect
Inspect `ref`, `action`, `active_nodes`, `relative_error_vs_ttno`, `time_env_build_sec`, `time_expr_build_sec`, `time_term_loop_sec`, `time_apply_sec`, and `time_total_sec`.

## 6. Answer prompt
**Your answer:** Why do tiny relative errors establish numerical identity but not make this measurement a full TDVP step or a full-state `H|psi>` apply? ______

## 7. Expected observation
The rows contain exactly `sop_no_env`, `sop_mctdh_like_state_env`, and `ttno_with_env`; all statuses are `ok` and maximum relative error is below $10^{-10}$. Every row names quantity `local_effective_1site_apply_all_nodes`.

## 8. Three-sentence oral explanation
All three paths produce the same collection of local effective-Hamiltonian actions to numerical precision. Their timing decomposition separates environment construction, expression construction, and application while keeping one shared total definition. The result compares an all-node local-action kernel, not state propagation over a timestep and not a full-state operator application.


In [ ]:
from types import SimpleNamespace
from benchmarks.benchmark_li2024_operator_env import run_li2024_point

if RUN_BREAKPOINTS:
    breakpoint()
rows = run_li2024_point(
    panel="primitive_basis",
    n_modes=4,
    state_bond=2,
    primitive_basis=3,
    contract_primitive=True,
    repeat_id=0,
    args=SimpleNamespace(active_scope="all_nodes", timeout_sec=120, memory_limit_mb=0.0),
    git_commit="sop-debugging-lab",
)
assert {row["method"] for row in rows} == {
    "sop_no_env", "sop_mctdh_like_state_env", "ttno_with_env"
}
assert max(row["relative_error_vs_ttno"] for row in rows) < 1e-10
timing_fields = [
    "time_env_build_sec", "time_expr_build_sec", "time_term_loop_sec",
    "time_apply_sec", "time_total_sec",
]
for row in sorted(rows, key=lambda item: item["method"]):
    print(row["method"], row["status"], row["quantity"],
          "relative error:", row["relative_error_vs_ttno"],
          {field: row[field] for field in timing_fields})


# Investigation 10 — Reproducing One Formal Data Point

## 1. Supervisor question
How can one measured method/repeat be traced from an immutable manifest record to a recoverable snapshot and then into a fit?

## 2. Formula
$\text{task}=(\text{panel},m,r,N_b,M_s,d,\text{topology})$, followed by $\log t=\log C+\alpha\log x$ after repeat aggregation.

## 3. Runnable observation
The next cell selects one real formal task, prints its dataclass fields and deterministic snapshot path, and prints the production functions that run, finalize, summarize, fit, and plot it without writing any files.

## 4. Exact breakpoint
In `benchmarks/run_li2024_formal_point.py::run_task`, stop after `rows = run_li2024_point(...)` and before `write_snapshot_atomic(output, payload)`.

## 5. Inspect
Inspect `task`, `output`, `rows[0]`, `payload`, the `status` field, and how `task_id` plus `repeat_id` determine the NPZ filename.

## 6. Answer prompt
**Your answer:** For each panel, what changes physically when $N_b$, $M_s$, or $d$ changes, and which field becomes the fit x-axis? ______

## 7. Expected observation
A task fixes exactly one panel/method/repeat and has a deterministic `.npz` path. The finalizer accepts only `status=ok` task IDs before summary/fits generation; `N_b` changes bath modes, $M_s$ changes state bonds, and $d$ changes local primitive basis size/topology.

## 8. Three-sentence oral explanation
The manifest makes every formal point reproducible by fixing method, repeat, dimensions, and adaptive topology. The point runner writes one atomic NPZ so interruption cannot erase other completed tasks, and the finalizer checks completeness before aggregation. Summary rows average repeats, fits use the declared panel x-axis, and the plotter visualizes those fits without changing the raw timing identity.


In [ ]:
from dataclasses import asdict
from benchmarks.li2024_formal_manifest import formal_tasks
from benchmarks.run_li2024_formal_point import run_task, snapshot_path
from benchmarks.finalize_li2024_formal import collect_snapshots, missing_task_ids
from benchmarks.plot_li2024_operator_scaling import compute_fits, summarize

formal_task = next(
    task for task in formal_tasks(repeats=1)
    if task.panel == "modes" and task.method == "sop_mctdh_like_state_env"
)
formal_snapshot = snapshot_path(Path("snapshots"), formal_task)
if RUN_BREAKPOINTS:
    breakpoint()
print("manifest task:", asdict(formal_task))
print("snapshot path:", formal_snapshot)
print("runner:", run_task.__module__ + "." + run_task.__name__)
print("finalizer:", collect_snapshots.__module__ + "." + collect_snapshots.__name__,
      missing_task_ids.__module__ + "." + missing_task_ids.__name__)
print("aggregate/fit:", summarize.__module__ + "." + summarize.__name__,
      compute_fits.__module__ + "." + compute_fits.__name__)


# Oral Recap and Formula/Code Lookup

## Completed measured results

- Mode-count largest-four-point exponents for `local_effective_1site_apply_all_nodes`: `3.109` (`sop_no_env`), `2.035` (`sop_mctdh_like_state_env`), and `0.916` (`ttno_with_env`).
- Full-workflow state-bond tail exponents: about `2.180`, `2.243`, and `2.223`; separately, the isolated full-rank internal contraction has mathematical $M_s^4$ FLOPs.
- Large-$d$ formal timings are nearly constant after adaptive primitive contraction; separately, isolated paired and contracted leaves expose $d^4$ and $d^2$ mathematical/storage behavior.

Measured finite-window wall-time exponents and isolated tensor-complexity powers answer different questions; do not relabel one as the other.

## Formula-to-code-object lookup

| Formula/object | Production code |
|---|---|
| $H=\sum_l c_l\prod_\kappa h_l^{(\kappa)}$ | `Op`, `SOPTerm`, `SOPBaselineOperator` |
| dof $\rightarrow$ tree node | `BasisTree.dof2idx`, `_split_op_by_tree_node` |
| $\langle i|h|j\rangle$ | `BasisSet.op_mat`, `_local_matrix_factors` |
| independent $H_l|\Psi\rangle$ | `_apply_term_to_ttns`, `apply_to_ttns_no_env` |
| $E_{l,u\rightarrow v}$ | `SOPMCTDHSweepEnvironment._env_cache` |
| operator-signature reuse | `SOPOneSiteEffective.branch_signature` |
| operator-bond compression | `TTNO`, `TTNEnviron`, `hop_expr1` |
| $\epsilon_m$ and timing rows | `run_li2024_point` |
| $t(x)=Cx^\alpha$ | `summarize`, `compute_fits`, plotter |

## What changed / Why / Effect

| What changed | Why | Effect observed |
|---|---|---|
| Added explicit `sop_no_env` identity | Expose repeated flat-term/tree work | Mode tail exponent `3.109` |
| Added strict directed-edge state messages keyed by term | Fairly reuse state contractions without cross-term operator sharing | Mode tail exponent reduced to `2.035` |
| Compared with TTNO operator bonds plus TTN environments | Measure joint state- and operator-structure reuse | Mode tail exponent `0.916` |
| Added adaptive primitive contraction | Avoid paired multi-mode leaf growth when $d>M_s$ | Formal large-$d$ timing nearly constant in the measured workflow |
| Added atomic per-task snapshots and completeness checks | Preserve completed repeats and prevent partial results being labeled final | Recoverable 189/189 formal workflow |

## Supervisor-facing three-sentence recap

We separated a naive flat SOP path, a strict MCTDH-like path that reuses only per-term state environments, and a TTNO path that also compresses repeated operator structure. On the same all-node one-site local-action quantity, their completed mode-count tail exponents are approximately 3.109, 2.035, and 0.916 with numerical agreement below the benchmark tolerance. State-bond and primitive-basis observations remain explicitly separated from isolated $M_s^4$, $d^4$, and $d^2$ tensor-complexity results because the measured workflow and mathematical kernels are not identical objects.

## Common mistakes

- Calling a physical bath a contraction environment.
- Calling `sop_env_plus_operator_cache` the strict MCTDH-like baseline; its `signature_metadata` says it is optimized, explanatory, and non-strict.
- Treating `TTNO.apply()` as the `TTNEnviron` local-effective-Hamiltonian path.
- Calling `local_effective_1site_apply_all_nodes` a complete TDVP propagation step or full-state `H|psi>`.
- Reporting an exponent without its x-axis, active scope, timing object, and fit window.
- Treating a finite-window wall-time exponent as an exact asymptotic FLOP power.

## Blank answer prompts

**What changed?**  
_Write your answer here:_

**Why was the strict state environment needed?**  
_Write your answer here:_

**What effect did it have on the measured mode-count scaling?**  
_Write your answer here:_

**Why is the signature cache explanatory rather than formal?**  
_Write your answer here:_

**What is the exact scientific timing identity?**  
_Write your answer here:_


In [ ]:
LAB_COMPLETE = True
print("SOP debugging lab completed successfully.")
